# Tool use & function calling

Tool use lets an LM delegate exact operations to external functions and fold results back into text.

This lesson turns transformer probabilities into a control loop: route to a tool, emit schema-constrained arguments, read the observation, and decide whether to retry or answer. Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

random.seed(9)
np.random.seed(9)

RUNG_NAMES = ["D1", "D2", "D3", "D4", "D5"]


def softmax(logits):
    values = np.array(logits, dtype=float)
    shifted = values - values.max()
    weights = np.exp(shifted)
    return weights / weights.sum()


def tokenize(text):
    clean = "".join(ch.lower() if ch.isalnum() else " " for ch in text)
    return [tok for tok in clean.split() if tok]


def cosine(a, b):
    left = np.array(a, dtype=float)
    right = np.array(b, dtype=float)
    denom = np.linalg.norm(left) * np.linalg.norm(right)
    if denom == 0:
        return 0.0
    return float(np.dot(left, right) / denom)


def bow_vector(text, vocab):
    counts = Counter(tokenize(text))
    return np.array([counts.get(term, 0) for term in vocab], dtype=float)


def preview_ladder(ladder):
    for rung in ladder:
        name = rung["name"]
        size = len(rung.get("tasks", rung.get("items", rung.get("queries", []))))
        note = rung.get("note", "")
        print(f"{name}: size={size} {note}")
        sample_key = "tasks" if "tasks" in rung else "items" if "items" in rung else "queries"
        print(rung[sample_key][0])


## The concept, built once

The lesson formula is $p(a\mid x)=p(\mathrm{tool}\mid x)\,p(\mathrm{args}\mid x,\mathrm{tool})$. We first make the routing distribution explicit. The lesson's logits $[2,1,0]$ become probabilities, while schema completeness and semantic argument error stay outside the language model so they can be checked exactly.

In [ ]:

def calculator(args):
    return args["left"] + args["right"]


def lookup(args):
    table = {"planets": 8, "continents": 7, "days": 7}
    return table.get(args["key"], None)


def convert_units(args):
    if args["from"] == "m" and args["to"] == "cm":
        return args["value"] * 100
    if args["from"] == "cm" and args["to"] == "m":
        return args["value"] / 100
    return None


TOOLS = {
    "calculator": calculator,
    "lookup": lookup,
    "unit_converter": convert_units,
}


SCHEMAS = {
    "calculator": ["left", "right", "operation"],
    "lookup": ["key", "source", "date"],
    "unit_converter": ["value", "from", "to"],
}


def validate_schema(tool, args):
    required = SCHEMAS[tool]
    present = [field for field in required if field in args]
    return len(present) / len(required)


The reusable policy chooses the highest-probability tool, validates syntax, executes a deterministic function, and folds the observation back in as an odds multiplier. A semantic validator catches cases where a schema-valid call is still the wrong call.

In [ ]:

def semantic_valid(tool, args, task):
    text = task["prompt"].lower()
    if tool == "calculator":
        return any(word in text for word in ["sum", "add", "total"])
    if tool == "lookup":
        return any(word in text for word in ["lookup", "how many", "known fact"])
    if tool == "unit_converter":
        return any(word in text for word in ["convert", "cm", "meters"])
    return False


def tool_policy(task, retry_cap=2, use_semantic_validator=True):
    probs = softmax(task["tool_logits"])
    order = list(np.argsort(-probs))
    attempts = []
    observation = None
    selected_tool = None
    selected_args = None
    for attempt_id, tool_index in enumerate(order[: retry_cap + 1]):
        tool = task["tools"][tool_index]
        args = task["candidate_args"][tool]
        completeness = validate_schema(tool, args)
        schema_ok = completeness == 1.0
        semantic_ok = semantic_valid(tool, args, task)
        allowed = schema_ok and (semantic_ok or not use_semantic_validator)
        attempts.append({"tool": tool, "schema_ok": schema_ok, "semantic_ok": semantic_ok})
        if allowed:
            selected_tool = tool
            selected_args = args
            observation = TOOLS[tool](args)
            break
    if observation is None:
        success = False
    else:
        success = observation == task["expected"]
    odds_multiplier = math.exp(task.get("observation_boost", 0.0))
    return {
        "tool_probs": probs,
        "selected_tool": selected_tool,
        "selected_args": selected_args,
        "observation": observation,
        "attempts": attempts,
        "success": success,
        "odds_multiplier": odds_multiplier,
    }


lesson_task = {
    "prompt": "Add 20 and 22, then answer with the exact total.",
    "tools": ["calculator", "lookup", "unit_converter"],
    "tool_logits": [2, 1, 0],
    "candidate_args": {
        "calculator": {"left": 20, "right": 22},
        "lookup": {"key": "planets"},
        "unit_converter": {"value": 2, "from": "m", "to": "cm"},
    },
    "expected": 42,
    "observation_boost": 1.2,
}

lesson_result = tool_policy(lesson_task, retry_cap=2, use_semantic_validator=False)
lesson_completeness = validate_schema("calculator", lesson_task["candidate_args"]["calculator"])
lesson_error = abs(42 - 40)
lesson_success_rate = 1 / 3

assert round(lesson_completeness, 3) == 0.667
assert lesson_error == 2
assert round(lesson_result["odds_multiplier"], 3) == 3.320
assert round(lesson_success_rate, 3) == 0.333
print(lesson_result)


## The dataset ladder

Build D1-D5 inline so the notebook is self-contained in Colab. Each rung adds scale or a realistic failure mode while keeping CPU-only toy data.

In [ ]:

def make_tool_ladder():
    d1 = [{"prompt": "Add 20 and 22.", "tools": ["calculator", "lookup", "unit_converter"], "tool_logits": [2, 1, 0], "candidate_args": {"calculator": {"left": 20, "right": 22, "operation": "add"}, "lookup": {"key": "days", "source": "calendar", "date": "today"}, "unit_converter": {"value": 2, "from": "m", "to": "cm"}}, "expected": 42, "observation_boost": 1.2}]
    d2 = d1 + [{"prompt": "Convert 3 meters to centimeters.", "tools": ["unit_converter", "calculator", "lookup"], "tool_logits": [2.4, 1.2, 0.1], "candidate_args": {"calculator": {"left": 3, "right": 100, "operation": "multiply"}, "lookup": {"key": "days", "source": "calendar", "date": "today"}, "unit_converter": {"value": 3, "from": "m", "to": "cm"}}, "expected": 300, "observation_boost": 0.8}]
    d3 = d2 + [{"prompt": "How many planets are in the Solar System?", "tools": ["calculator", "lookup", "unit_converter"], "tool_logits": [2.1, 2.0, 0.3], "candidate_args": {"calculator": {"left": 4, "right": 4, "operation": "add"}, "lookup": {"key": "planets", "source": "almanac", "date": "current"}, "unit_converter": {"value": 8, "from": "m", "to": "cm"}}, "expected": 8, "observation_boost": 1.0}]
    d4 = d3 + [{"prompt": "A report says 7 continents and asks for a known fact lookup.", "tools": ["lookup", "calculator", "unit_converter"], "tool_logits": [1.8, 1.6, 0.2], "candidate_args": {"calculator": {"left": 3, "right": 4, "operation": "add"}, "lookup": {"key": "continents", "source": "atlas", "date": "current"}, "unit_converter": {"value": 7, "from": "m", "to": "cm"}}, "expected": 7, "observation_boost": 0.9}]
    d5 = d4 + [{"prompt": "Long memo: a distracting schema-valid calculator call appears, but the actual request is to convert 2 m to cm.", "tools": ["calculator", "unit_converter", "lookup"], "tool_logits": [2.2, 2.0, 0.1], "candidate_args": {"calculator": {"left": 40, "right": 2, "operation": "add"}, "lookup": {"key": "days", "source": "calendar", "date": "today"}, "unit_converter": {"value": 2, "from": "m", "to": "cm"}}, "expected": 200, "observation_boost": 1.1}]
    return [
        {"name": "D1", "tasks": d1, "note": "one prompt"},
        {"name": "D2", "tasks": d2, "note": "few-shot tools"},
        {"name": "D3", "tasks": d3, "note": "ambiguous schemas"},
        {"name": "D4", "tasks": d4, "note": "calculator lookup converter"},
        {"name": "D5", "tasks": d5, "note": "long context with loop risk"},
    ]


tool_ladder = make_tool_ladder()
preview_ladder(tool_ladder)


## Run the same method across D1-D5

The metric follows the plan: task success, retrieval recall, unsupported rate, benchmark accuracy, or safety pass-rate depending on the topic.

In [ ]:

tool_rows = []
for rung in tool_ladder:
    successes = []
    for task in rung["tasks"]:
        result = tool_policy(task, retry_cap=2, use_semantic_validator=True)
        successes.append(float(result["success"]))
    success_rate = sum(successes) / len(successes)
    tool_rows.append({"rung": rung["name"], "metric": success_rate, "n": len(successes)})

for row in tool_rows:
    print(row)


## Results visualization

The closing figure uses small multiples for the per-rung artifact and one summary curve across D1-D5.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for index, rung in enumerate(tool_ladder):
    task = rung["tasks"][-1]
    result = tool_policy(task, retry_cap=2, use_semantic_validator=True)
    labels = [attempt["tool"] for attempt in result["attempts"]]
    values = [1 if attempt["semantic_ok"] else 0 for attempt in result["attempts"]]
    axes[0, index].bar(labels, values, color="steelblue")
    axes[0, index].set_title(rung["name"])
    axes[0, index].set_ylim(0, 1.1)
    axes[0, index].tick_params(axis="x", rotation=45)

axes[1, 0].plot([row["rung"] for row in tool_rows], [row["metric"] for row in tool_rows], marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_ylabel("task success")
axes[1, 0].set_title("success vs rung")
for blank in axes[1, 1:]:
    blank.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Reproduce the named D5 failure, then turn on the fix and compare the behavior.

In [ ]:

d5_task = tool_ladder[-1]["tasks"][-1]
wrong = tool_policy(d5_task, retry_cap=5, use_semantic_validator=False)
fixed = tool_policy(d5_task, retry_cap=2, use_semantic_validator=True)
print("schema-only selected", wrong["selected_tool"], wrong["observation"], wrong["success"])
print("semantic validator selected", fixed["selected_tool"], fixed["observation"], fixed["success"])


## Evaluate it + Practice

- Compare the planned metric with a no-skill baseline such as answering without tools, retrieval, claims, intervals, or guardrails.
- Sanity-check one hand-computable lesson number before trusting the ladder.
- Ablate the key idea and verify the metric drops or the failure signal rises.
- Watch failure signals: retry loops, irrelevant evidence, hidden unsupported claims, noisy rank changes, or unsafe tool actions.

Practice prompts:
- Change the semantic validator so multiplication requests route to the calculator without weakening schema checks.
- Add a calendar lookup task and compare success with retry caps 0, 1, and 2.
- Create one adversarial D5 prompt where the first schema-valid call is wrong, then show the validator blocks it.
